In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from recommenders.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float64

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [3]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [4]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100

In [5]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity")
R = torch.tensor(matrix_mf.R, dtype=dtype, device=device)
R

tensor([[ 0.,  0., 62.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0., 54.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [44., 60.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ..., 92.,  0.,  0.]], device='mps:0')

In [6]:
alpha = matrix_mf.compute_alpha()
R *= alpha
alpha

0.13540265635507734

In [7]:
lmf = LogisticMatrixFactorization(
    R=R,
    num_factors=3,
    alpha=alpha,
    lambd=0.01,
    device=device,
    dtype=dtype,
)

num_epochs = 1000
lmf.train_with_gradients(
    num_epochs=num_epochs,
    learning_rate=0.01,
    log_interval=10,
)

Epoch 1: loss = 4347.703125, MPR = 0.4925605058670044
Epoch 11: loss = 3938.254638671875, MPR = 0.47927209734916687
Epoch 21: loss = 3790.806640625, MPR = 0.4692744314670563
Epoch 31: loss = 3673.7470703125, MPR = 0.45993542671203613
Epoch 41: loss = 3576.69482421875, MPR = 0.4515012800693512
Epoch 51: loss = 3494.406982421875, MPR = 0.4434061646461487
Epoch 61: loss = 3423.61181640625, MPR = 0.4354008436203003
Epoch 71: loss = 3362.043212890625, MPR = 0.42772936820983887
Epoch 81: loss = 3308.036376953125, MPR = 0.42047563195228577
Epoch 91: loss = 3260.319580078125, MPR = 0.41344597935676575
Epoch 101: loss = 3217.894775390625, MPR = 0.40692228078842163
Epoch 111: loss = 3179.962890625, MPR = 0.4005732536315918
Epoch 121: loss = 3145.873291015625, MPR = 0.39454859495162964
Epoch 131: loss = 3115.089599609375, MPR = 0.3887907862663269
Epoch 141: loss = 3087.16552734375, MPR = 0.3834521174430847
Epoch 151: loss = 3061.725830078125, MPR = 0.37837544083595276
Epoch 161: loss = 3038.45288

In [ ]:
lmf.save("models", "lmf_3")
lmf = LogisticMatrixFactorization.load(os.path.join("models", "lmf_3.pt"))

In [ ]:
px.line(x=range(len(lmf.mprs)), y=lmf.mprs.cpu()).show()
px.line(x=range(len(lmf.losses)), y=lmf.losses.cpu()).show()

In [ ]:
user_id = matrix_mf.usernames_to_ids(["paul"])[0]

# Get the top 10 recommendations for the user
top_10_ids, top_10_ids_scores = lmf.recommend(user_id, top_k=20, filter_user_items=True)

# Get the top 10 recommendations for the user
matrix_mf.items_ids_to_df(top_10_ids)[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
0,jaslkh,ROSALÍA,PROMESA,2023,63,0.630,0.374,0.2270,0.86100,0.000075,0.2940,0.3730,104.955,-9.007,196426,2023,63
50,jaslkh,Jain,Makeba,2016,80,0.824,0.656,0.0704,0.38800,0.506000,0.2530,0.4180,116.068,-9.432,249533,2016,80
1605,owen,Josman,Goal,2021,50,0.822,0.590,0.4200,0.46200,0.000000,0.0904,0.5530,113.963,-8.981,281052,2021,50
1614,owen,Iliona,micha,2022,49,0.795,0.417,0.0364,0.67900,0.573000,0.1090,0.4460,121.989,-12.028,233413,2022,49
1615,owen,Iliona,Si tu m'aimes demain,2022,59,0.854,0.501,0.0310,0.63200,0.001190,0.1030,0.4650,105.011,-9.535,176026,2022,59
1616,owen,Iliona,Garçon manqué,2022,46,0.822,0.399,0.0381,0.69300,0.101000,0.1030,0.3990,119.981,-9.382,211386,2022,46
6967,brenda,Angèle,Oui ou non,2019,62,0.649,0.574,0.1430,0.63500,0.000013,0.1610,0.3610,199.906,-7.856,196800,2019,62
6970,brenda,Taylor Swift,I Knew You Were Trouble.,2012,82,0.622,0.469,0.0363,0.00454,0.000002,0.0335,0.6790,77.019,-6.798,219720,2012,82
6980,brenda,Imagine Dragons,Bones,2022,86,0.772,0.750,0.0455,0.02010,0.000000,0.0740,0.5870,114.061,-3.670,165264,2022,86
6981,brenda,Imagine Dragons,Enemy (with JID) - from the series Arcane Leag...,2021,81,0.728,0.783,0.2660,0.23700,0.000000,0.4340,0.5550,77.011,-4.424,173381,2021,81


In [ ]:
df_tracks = df_matrix_mf.copy()
df_tracks = df_tracks.sample(frac=1) # Shuffle the dataframe
df_tracks = df_tracks[:1000] # Keep only 1000 tracks

# Add item latent factors to the dataframe
items_latent_columns = [f"track_latent_{i}" for i in range(lmf.num_factors)]
for track_id in df_tracks["id"].unique():
    latent_factors = lmf.get_item_latent_factors(matrix_mf.items_to_ids([track_id])[0]).tolist()
    df_tracks.loc[df_tracks["id"] == track_id, items_latent_columns] = latent_factors

# Add user latent factors to the dataframe
users_latent_columns = [f"latent_factor_{i}" for i in range(lmf.num_factors)]
for user_id in df_tracks["username"].unique():
    latent_factors = lmf.get_user_latent_factors(matrix_mf.usernames_to_ids([user_id])[0]).tolist()
    df_tracks.loc[df_tracks["username"] == user_id, users_latent_columns] = latent_factors

plotting.plot_latent_space(
    df=df_tracks,
    color=df_tracks["username"],
    text=df_tracks["username"],
    latent_columns=items_latent_columns,
).show()

plotting.plot_latent_space(
    df=df_tracks,
    color=df_tracks["username"],
    text=df_tracks["username"],
    latent_columns=users_latent_columns,
).show()